# 19. MCP

MCP는 프로세스 밖의 도구를 검색하고 호출하며 결과를 전달하는 방식을 표준화한다. 같은 서버에 네 가지 하네스를 연결해 배선과 계측 비용을 비교한다.

### 공통 설정

Part 1 `1`과 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [1]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}  # 1M 토큰당 (입력, 출력) USD
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 서버와 버전 제약

`../mcp/corpus_server.py`는 도구 함수를 MCP 서버로 노출하는 실행 파일이다. 독스트링은 도구 스키마 설명으로 사용된다.

`langchain-mcp-adapters`가 `mcp<2`를 요구하므로 이 과정은 `mcp==1.29.1`을 사용한다. mcp 2.x에서는 `FastMCP`가 `MCPServer`로 바뀌었으며 별도 패키지 `fastmcp`와 혼동하지 않는다.

In [3]:
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.streamable_http import streamable_http_client

SERVER = Path("../mcp/corpus_server.py")
STDIO = StdioServerParameters(command=sys.executable, args=[str(SERVER)])
PORT = 8931
URL = f"http://127.0.0.1:{PORT}/mcp"
Q = "'한국독립당' 이 나오는 문서가 몇 편인지 세고 숫자로 답한다"

서버 파일의 앞부분이다. 도구 하나가 함수 하나다.

In [4]:
print(SERVER.read_text(encoding="utf-8")[:900])

"""코퍼스 도구 셋을 MCP 로 내보내는 서버.

노트북이 아니라 실행 파일이다. FastMCP 는 mcp SDK 가 제공하는 서버 쪽 편의 API 다.
도구 하나가 함수 하나이고, 이름과 독스트링과 타입 힌트가 그대로 스키마가 된다.

  python "mcp/corpus_server.py"                 stdio. 클라이언트가 자식 프로세스로 띄운다
  python "mcp/corpus_server.py" http 8931       streamable-http. 따로 떠 있고 여럿이 붙는다

두 전송의 차이와 그에 따르는 결정은 19 노트북에 있다.
"""

import sys
from pathlib import Path

from mcp.server.fastmcp import FastMCP

CORPUS = Path(__file__).resolve().parent.parent.parent / "data" / "corpus" / "graph"

TRANSPORT = "streamable-http" if len(sys.argv) > 1 and sys.argv[1] == "http" else "stdio"
PORT = int(sys.argv[2]) if len(sys.argv) > 2 else 8931

server = FastMCP("corpus", host="127.0.0.1", port=PORT)


@server.tool()
def list_docs() -> list[str]:
    """코퍼스에 있는 문서 제목을 모두 돌려준다."""
    return [f.stem for f in sorted(CORPUS.glob("*.md"))]


@server.tool()
def grep_docs(pattern: str) -> list[dict]:
    """낱


### 도구 검색

stdio 전송은 서버를 자식 프로세스로 띄운다. 세션을 열고 함수 하나를 실행한 뒤 닫는 도우미를 둔다.

In [5]:
async def with_stdio(fn):
    async with stdio_client(STDIO) as (r, w):   # stdio에 read, write로
        async with ClientSession(r, w) as session:
            await session.initialize()
            return await fn(session)

`list_tools`는 도구 이름, 설명, 인자 스키마를 반환한다.

In [6]:
tools = await with_stdio(lambda s: s.list_tools())
print([t.name for t in tools.tools])

['list_docs', 'grep_docs', 'read_doc', 'read_doc_raising']


도구마다 설명과 입력 스키마를 읽는다.

In [7]:
for t in tools.tools:
    print(f"{t.name:18} {t.description[:32]}")
    print(f"                   {json.dumps(t.inputSchema, ensure_ascii=False)[:84]}")

list_docs          코퍼스에 있는 문서 제목을 모두 돌려준다.
                   {"properties": {}, "title": "list_docsArguments", "type": "object"}
grep_docs          낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다.
                   {"properties": {"pattern": {"title": "Pattern", "type": "string"}}, "required": ["pa
read_doc           문서 한 편의 앞부분을 돌려준다.
                   {"properties": {"title": {"title": "Title", "type": "string"}}, "required": ["title"
read_doc_raising   비교용. 없는 제목에 예외를 던진다. 하네스에는 넘기지 않
                   {"properties": {"title": {"title": "Title", "type": "string"}}, "required": ["title"


### Responses API 스키마로 변환

Responses API의 `strict` 모드는 MCP의 JSON Schema보다 좁다. `additionalProperties: false`를 지정하고 모든 속성을 `required`에 넣어야 한다.

In [8]:
def to_openai(tool):
    props = dict(tool.inputSchema).get("properties", {})
    return {"type": "function", "name": tool.name, "strict": True,
            "description": tool.description,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}

비교용 `read_doc_raising`은 빼고 변환한다.

In [9]:
TOOLS = [to_openai(t) for t in tools.tools if t.name != "read_doc_raising"]
print([t["name"] for t in TOOLS])
print(json.dumps(TOOLS[1], ensure_ascii=False)[:150])

['list_docs', 'grep_docs', 'read_doc']
{"type": "function", "name": "grep_docs", "strict": true, "description": "낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다.", "parameters": {"type": "object", "propertie


### 결과 블록 처리

목록 결과는 항목별 블록으로 나뉠 수 있어 `content[0]`만 읽으면 일부가 누락된다. 원래 구조가 필요하면 `structuredContent`를 사용한다.

In [10]:
def body_of(out):
    if out.structuredContent is not None:
        return json.dumps(out.structuredContent["result"], ensure_ascii=False)
    return "\n".join(c.text for c in out.content)

`grep_docs`를 한 번 부른다.

In [11]:
got = await with_stdio(lambda s: s.call_tool("grep_docs", {"pattern": "한국독립당"}))
print(body_of(got)[:120])

[]


블록 수와 `structuredContent`로 읽은 건수를 비교한다.

In [12]:
print(f"블록 {len(got.content)}개   structuredContent 로 읽으면 {len(json.loads(body_of(got)))}건")

블록 0개   structuredContent 로 읽으면 0건


### 버전별 오류 전달

서버 예외의 문구는 MCP 버전에 따라 달라질 수 있다. 복구에 필요한 오류 정보는 예외보다 반환값으로 전달하는 편이 안정적이다.

In [13]:
async def probe(session):
    return (await session.call_tool("read_doc", {"title": "없는문서"}),
            await session.call_tool("read_doc_raising", {"title": "없는문서"})) #exception 명시적 발생

없는 제목으로 둘을 부른다. 값으로 돌려준 쪽과 예외로 던진 쪽을 비교한다.

In [ ]:
ret, rai = await with_stdio(probe)
print(f"값으로 반환  isError {ret.isError}  {body_of(ret)[:56]}")
print(f"예외로 던짐  isError {rai.isError}  {body_of(rai)[:56]}")

이 판은 예외 문구를 실어 보내지만 `isError`와 접두어가 붙는다. mcp 2.x에서 같은 코드를 돌리면 문구가 사라지고 실패했다는 사실만 온다. 값으로 돌려주면 어느 판에서도 그대로 도착한다.

### 전송 방식 선택

| | stdio | streamable-http |
|---|---|---|
| 서버 실행 | 클라이언트의 자식 프로세스 | 별도 프로세스 |
| 권한 경계 | 프로세스 경계 | 별도 인증 필요 |
| 클라이언트 | 각자 실행 | 서버 공유 |
| 인자 검증 | 같은 사용자 경계 안 | 서버에서 수행 |

HTTP 서버는 클라이언트를 신뢰할 수 없으므로 경로 검증과 인증을 서버에 둬야 한다.

HTTP 서버를 별도 프로세스로 띄우고 포트가 열릴 때까지 기다린다. 마지막 절에서 내린다.
</br> => AI Agent가 외부의 도구를 사용하기 위함!!

In [ ]:
import socket
import subprocess

proc = subprocess.Popen([sys.executable, str(SERVER), "http", str(PORT)],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=0.3).close()
        break
    except OSError:
        time.sleep(0.3)

HTTP 세션용 도우미다. stdio 쪽과 같은 모양이다.

In [ ]:
async def with_http(fn):
    async with streamablehttp_client(URL) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            return await fn(session)

같은 서버라 도구 목록도 같다.

In [ ]:
print([t.name for t in (await with_http(lambda s: s.list_tools())).tools])

호출당 지연을 잰다. 다섯 번 부른 평균이다.

In [ ]:
async def bench(session):
    t = time.perf_counter()
    for _ in range(5):
        await session.call_tool("grep_docs", {"pattern": "한국독립당"})
    return (time.perf_counter() - t) / 5 * 1000

stdio는 매번 서버를 띄우고 http는 한 번 띄워 둔다. 서버를 띄우는 시간은 아래 값에 들어 있지 않다.

In [ ]:
print(f"stdio {await with_stdio(bench):5.1f}ms   http {await with_http(bench):5.1f}ms   호출당")

### 같은 서버에 네 가지 배선

서버와 질문을 고정하고 클라이언트 배선만 비교한다. 지시문과 반복 예산은 네 배선에 같이 준다.

In [ ]:
CONFIG = {"instructions": "도구로 확인한 것만 답한다.", "max_turns": 6}

자작 하네스는 세션 안에서 도구 목록을 받아 변환하고 루프를 직접 돈다.

In [ ]:
async def wire_own(question):
    async with streamablehttp_client(URL) as (r, w, _):
        async with ClientSession(r, w) as session:
            await session.initialize()
            spec = [to_openai(t) for t in (await session.list_tools()).tools
                    if t.name != "read_doc_raising"]
            messages = [{"role": "user", "content": question}]
            for _ in range(CONFIG["max_turns"]):
                resp = keep(client.responses.create(
                    model=MODEL, instructions=CONFIG["instructions"],
                    input=messages, tools=spec, store=False))
                calls = [i for i in resp.output if i.type == "function_call"]
                if not calls:
                    return resp.output_text, messages
                messages += [i.model_dump() for i in resp.output]
                for c in calls:
                    out = await session.call_tool(c.name, json.loads(c.arguments))
                    messages.append({"type": "function_call_output",
                                     "call_id": c.call_id, "output": body_of(out)})
            return None, messages

질문을 돌린다. 입력 배열도 같이 돌려받아 뒤에서 계측에 쓴다.

In [ ]:
own_answer, own_log = await wire_own(Q)
print("자작 하네스 |", own_answer)

Agents SDK는 MCP 서버를 에이전트에 붙이면 도구 변환과 루프를 대신한다.

In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp


async def wire_agents_sdk(question):
    async with MCPServerStreamableHttp(params={"url": URL}) as srv:
        agent = Agent(name="corpus", model=MODEL,
                      instructions="도구로 확인한 것만 답한다.", mcp_servers=[srv])
        return await Runner.run(agent, question)

같은 질문을 돌린다.

In [ ]:
sdk_result = await wire_agents_sdk(Q)
print("Agents SDK  |", sdk_result.final_output)

LangGraph는 어댑터가 MCP 도구를 LangChain 도구로 바꾼다.

In [ ]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient


async def wire_langgraph(question):
    mcp_client = MultiServerMCPClient({"corpus": {"url": URL, "transport": "streamable_http"}})
    agent = create_agent(f"openai:{MODEL}", await mcp_client.get_tools(),
                         system_prompt="도구로 확인한 것만 답한다.")
    return await agent.ainvoke({"messages": [("user", question)]})

같은 질문을 돌린다. 마지막 메시지가 답이다.

In [ ]:
lg_result = await wire_langgraph(Q)
print("LangGraph   |", lg_result["messages"][-1].content)

코딩 에이전트는 설정 파일 한 덩이다. 코드가 없다.

In [ ]:
def wire_coding_agent():
    return {"mcpServers": {"corpus": {"type": "http", "url": URL}}}


print("코딩 에이전트 |", json.dumps(wire_coding_agent(), ensure_ascii=False))

설정 파일로 써 둔다. 마지막 절에서 지운다.

In [ ]:
Path("mcp_config.json").write_text(json.dumps(wire_coding_agent(), ensure_ascii=False, indent=1),
                                   encoding="utf-8")
print(Path("mcp_config.json").stat().st_size, "바이트")

### 배선 코드 비교

주석과 빈 줄을 제외한 클라이언트 배선만 센다. 공통 서버와 질문은 제외한다.

In [ ]:
import inspect


def lines(fn):
    body = inspect.getsource(fn).splitlines()
    return len([l for l in body if l.strip() and not l.strip().startswith("#")])

네 배선의 줄 수와 추가 의존성이다.

In [ ]:
rows = [("자작 하네스", lines(wire_own) + lines(to_openai), "없음"),
        ("Agents SDK", lines(wire_agents_sdk), "openai-agents"),
        ("LangGraph", lines(wire_langgraph), "langchain-mcp-adapters"),
        ("코딩 에이전트", lines(wire_coding_agent), "없음. 설정만")]

for name, n, dep in rows:
    print(f"{name:14} {n:3d}줄   추가 의존성 {dep}")

### 계측 비용 비교

도구 호출 목록, 오류 수, 턴 수를 추출하는 코드까지 비교해야 운영 비용을 판단할 수 있다. 세 배선의 실행 기록에서 같은 값을 꺼내는 함수다.

In [ ]:
def own_metrics(log):
    calls = [m["name"] for m in log if m.get("type") == "function_call"]
    outs = [m["output"] for m in log if m.get("type") == "function_call_output"]
    return {"도구": calls, "오류": sum(o.startswith("error") for o in outs), "턴": len(calls) + 1}


def sdk_metrics(result):
    calls = [i.raw_item.name for i in result.new_items if type(i).__name__ == "ToolCallItem"]
    outs = [str(i.output) for i in result.new_items if type(i).__name__ == "ToolCallOutputItem"]
    return {"도구": calls, "오류": sum(o.startswith("error") for o in outs), "턴": len(calls) + 1}


def lg_metrics(result):
    msgs = result["messages"]
    calls = [c["name"] for m in msgs for c in (getattr(m, "tool_calls", None) or [])]
    outs = [m.content for m in msgs if type(m).__name__ == "ToolMessage"]
    return {"도구": calls, "오류": sum(str(o).startswith("error") for o in outs), "턴": len(calls) + 1}

같은 계측을 세 배선에 적용한다. 코딩 에이전트는 실행이 우리 프로세스가 아니라 꺼낼 수 없다.

In [ ]:
print("자작        ", own_metrics(own_log), f"{lines(own_metrics)}줄")
print("Agents SDK  ", sdk_metrics(sdk_result), f"{lines(sdk_metrics)}줄")
print("LangGraph   ", lg_metrics(lg_result), f"{lines(lg_metrics)}줄")

### 비교 결과

| 배선 | 배선 줄 수 | 계측 추가 | 추가 의존성 | 제약 |
|---|---|---|---|---|
| 자작 하네스 | 가장 많다 | 몇 줄 | 없음 | 없음 |
| Agents SDK | 다섯 줄 안팎 | 몇 줄 | `openai-agents` | 반복 횟수 외의 예산은 별도 처리 |
| LangGraph | 다섯 줄 안팎 | 몇 줄 | `langchain-mcp-adapters` | 컨텍스트 절단 지점 없음 |
| 코딩 에이전트 | 설정만 | 불가 | 없음 | 자체 평가 코드에 연결 불가 |

자작 하네스, Agents SDK, LangGraph는 실행 기록에서 계측값을 비슷한 비용으로 추출할 수 있다. 코딩 에이전트는 실행이 별도 프로세스에 있어 같은 방식으로 계측할 수 없다.

평가와 개선 루프가 필요하면 배선 길이보다 궤적 추출과 제어 가능 여부를 먼저 확인한다. 의존성의 MCP 버전 제약과 버전별 오류 전달 차이도 함께 검토해야 한다.

HTTP 서버를 내리고 설정 파일을 지운다.

In [ ]:
proc.terminate()
proc.wait(timeout=10)
Path("mcp_config.json").unlink(missing_ok=True)
print(proc.returncode)

### 비용 확인

`keep`으로 모아 둔 usage에 단가를 곱해 합산한다. SDK와 LangGraph 배선의 사용량은 `keep`을 지나지 않으므로 여기에 없다. 그것도 감춰진 것 하나다.

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")